## Starting Cross-Lingual Evaluation

Now we will try to see how well the model performs between Scandinavian languages.

## Installing and Importing libraries



In [ ]:
!pip install transformers datasets seqeval torch -q

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
)
from datasets import load_dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

print('Imports OK')


## Loading datasets

Using multilingual/scandinavian NER datasets here.

In [ ]:


# Path to the English model saved by ner_baseline.ipynb
ENGLISH_MODEL_DIR = './ner_model_output'   #Update if needed

# Probably can optimize this later
#Target languages for zero-shot evaluation
LANGUAGES = {
    'da': 'Danish',
    'sv': 'Swedish',
    'no': 'Norwegian',
}

#Label scheme (must match the English Model exactly) 
LABEL_LIST  = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG']
LABEL_TO_ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

#Inference Settings 
EVAL_BATCH_SIZE = 16
MAX_LENGTH      = 512

print(f'English model dir: {ENGLISH_MODEL_DIR}')
print(f'Target languages : {list(LANGUAGES.values())}')
print(f'Label scheme     : {LABEL_LIST}')

## Tokenization

Subword tokenization made this part more complicated than expected.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

print(f'\nLoading English model from: {ENGLISH_MODEL_DIR}')
model     = AutoModelForTokenClassification.from_pretrained(ENGLISH_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(ENGLISH_MODEL_DIR)
model     = model.to(device)
model.eval()

#Sanity check: the model's label mapping should match ours
assert model.config.id2label[0] == 'O', 'Model label scheme looks wrong!'
print(f'Model loaded. Num labels: {model.config.num_labels}')
print(f'Labels: {list(model.config.id2label.values())}')

## Label alignment

Need labels to correctly match subtokens.

In [ ]:
raw_datasets = {}
for lang_code, lang_name in LANGUAGES.items():
    print(f'Loading WikiANN for {lang_name} ({lang_code})...')
    raw_datasets[lang_code] = load_dataset('wikiann', lang_code)

#Build the remapping table
    #WikiANN Label order (from the Dataset features):
wikiann_labels = raw_datasets['da']['train'].features['ner_tags'].feature.names
        #Checking Predictions Manually here
print(f'\nWikiANN label order: {wikiann_labels}')
print(f'Our model label order: {LABEL_LIST}')

    # Map:WikiANN integer ID → label string → our integer ID
WIKIANN_TO_OURS = {}
for wiki_id, label_name in enumerate(wikiann_labels):
    WIKIANN_TO_OURS[wiki_id] = LABEL_TO_ID[label_name]
print(f'Remap table: {WIKIANN_TO_OURS}')
    #E.G: WikiANN id 3 (B-ORG) → our id 5 (B-ORG)

#Now it will show test set sizes
print()
for lang_code, lang_name in LANGUAGES.items():
    n = len(raw_datasets[lang_code]['test'])
    print(f'  {lang_name:<10}: {n:>5,} test sentences')


## Loading model

Using multilingual transformer for token classification.

In [ ]:


def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
        padding=False,
    )
    all_labels = []
    for i, tag_ids in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned, prev_word_idx = [], None
        for word_idx in word_ids:
            if word_idx is None:
                aligned.append(-100)
            elif word_idx != prev_word_idx:
                #Remap WikiANN ID → our model's ID
                aligned.append(WIKIANN_TO_OURS[tag_ids[word_idx]])
            else:
                aligned.append(-100)
            prev_word_idx = word_idx
        all_labels.append(aligned)
    tokenized['labels'] = all_labels
    return tokenized


tokenized_tests = {}
for lang_code, lang_name in LANGUAGES.items():
    print(f'Tokenizing {lang_name} test set...')
    tokenized_tests[lang_code] = raw_datasets[lang_code]['test'].map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=raw_datasets[lang_code]['test'].column_names,
    )
    print(f'→ {len(tokenized_tests[lang_code])} examples ready for inference')

## Training setup

Trainer API makes training easier to manage.

In [ ]:
eval_args = TrainingArguments(
    output_dir='./cross_lingual_eval_tmp',

    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    fp16=(device.type == 'cuda'),
    report_to='none',
    do_train=False,
    do_eval=True,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=eval_args,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print('Inference trainer ready.')


## Running evaluation

Testing performance on other languages too.

In [ ]:
def predictions_to_label_sequences(predictions_output):
    """
# honestly transformers are still confusing sometimes
    Convert the Trainer's PredictionOutput → lists of true/pred label strings
    per sentence, ignoring -100 positions.
    """
    logits = predictions_output.predictions
    true_ids = predictions_output.label_ids
    pred_ids = np.argmax(logits, axis=-1)

    true_strs, pred_strs = [], []
    for pred_seq, true_seq in zip(pred_ids, true_ids):
        t, p = [], []
        for pid, tid in zip(pred_seq, true_seq):
            if tid == -100:
                continue
            t.append(ID_TO_LABEL[tid])
            p.append(ID_TO_LABEL[pid])
        true_strs.append(t)
        pred_strs.append(p)
    return true_strs, pred_strs


## Predictions and analysis

Trying to see where model performs badly.

In [ ]:


cross_lingual_results = {}

for lang_code, lang_name in LANGUAGES.items():
    print(f'\n{"="*60}')
    print(f'  ZERO-SHOT EVAL: English model → {lang_name.upper()}')
    print(f'{"="*60}\n')

    preds_output = trainer.predict(tokenized_tests[lang_code])
    true_strs, pred_strs = predictions_to_label_sequences(preds_output)

    f1  = f1_score(true_strs, pred_strs)
    pre = precision_score(true_strs, pred_strs)
    rec = recall_score(true_strs, pred_strs)

    cross_lingual_results[lang_code] = {
        'language': lang_name,
        'f1': f1,
        'precision': pre,
        'recall': rec,
        'true': true_strs,
        'pred': pred_strs,
    }

    print(f'F1        : {f1:.4f}')
    print(f'Precision : {pre:.4f}')
    print(f'Recall    : {rec:.4f}')
    print()
    print(classification_report(true_strs, pred_strs))




## Saving outputs

Saving results and trained model.

In [ ]:

summary_rows = []
for lang_code, res in cross_lingual_results.items():
    summary_rows.append({
        'Language':  res['language'],
        'Code':      lang_code,
        'F1':        round(res['f1'], 4),
        'Precision': round(res['precision'], 4),
        'Recall':    round(res['recall'], 4),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Language')
summary_df

## Per-Entity-Type Breakdown
The overall F1 hides where the model succeeds and fails. This table shows F1 for
each entity type (PER / LOC / ORG) across all three languages. Useful for spotting
patterns like "PER transfers well but ORG collapses".

In [ ]:


def per_type_f1(true_strs, pred_strs):
    """Extract per-entity-type F1 from seqeval's classification_report dict."""
    report = classification_report(true_strs, pred_strs, output_dict=True)
    return {etype: round(report.get(etype, {}).get('f1-score', 0.0), 4)
            for etype in ['PER', 'LOC', 'ORG']}


per_type_rows = []
for lang_code, res in cross_lingual_results.items():
    type_f1 = per_type_f1(res['true'], res['pred'])
    per_type_rows.append({
        'Language': res['language'],
        'PER F1':   type_f1['PER'],
        'LOC F1':   type_f1['LOC'],
        'ORG F1':   type_f1['ORG'],
        'Overall F1': round(res['f1'], 4),
    })

per_type_df = pd.DataFrame(per_type_rows).set_index('Language')
per_type_df


## Save Predictions to .iob2 Files
For each language, write the cross-lingual predictions to disk in the same
`word<TAB>pred_label` format as the baseline notebook. Useful for manual inspection
or further analysis (e.g., error analysis per entity type).

We map subword predictions back to word-level by taking the first subword's
prediction for each original word.

In [ ]:


for lang_code, lang_name in LANGUAGES.items():
    test_raw = raw_datasets[lang_code]['test']

    preds_output = trainer.predict(tokenized_tests[lang_code])
    pred_ids = np.argmax(preds_output.predictions, axis=-1)

    #Re-tokenize to get word_ids for subword → word mapping
# reducing batch size because memory issue before
    retokenized = tokenizer(
        [ex['tokens'] for ex in test_raw],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
        padding=False,
    )

    output_predictions = []
    for i, pred_seq in enumerate(pred_ids):
        word_ids = retokenized.word_ids(batch_index=i)
        sentence_preds, prev_word_idx = [], None
        for j, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue
            if word_idx != prev_word_idx:
                sentence_preds.append(ID_TO_LABEL[pred_seq[j]])
            prev_word_idx = word_idx
        output_predictions.append(sentence_preds)

    output_path = f'./crosslingual_predictions_en_to_{lang_code}.iob2'
    with open(output_path, 'w', encoding='utf-8') as f:
        for example, preds in zip(test_raw, output_predictions):
            for token, pred in zip(example['tokens'], preds):
                f.write(f'{token}\t{pred}\n')
            f.write('\n')

    print(f'Saved {lang_name} predictions → {output_path}')

print('\nAll cross-lingual predictions written to disk.')



